# 01 — PINN foundations: from PDE to computational graph

This notebook builds a Physics-Informed Neural Network (PINN) from first principles for the 1D heat equation

$$u_t - \alpha u_{xx}=0,\qquad x\in[-1,1],\;t\in[0,1]$$

with

$$u(x,0)=\sin(\pi x),\qquad u(-1,t)=u(1,t)=0.$$

The notebook implements the neural ansatz, input normalization, automatic differentiation, PDE residual, constraint losses, optimization, and analytical validation.

In [ ]:
import torch
from torch import nn
import matplotlib.pyplot as plt

torch.set_default_dtype(torch.float32)
torch.manual_seed(42)
alpha = 0.1


## 1. Neural-network solution ansatz

We approximate the unknown field with $u_\theta(x,t)$. Smooth activations are useful because the PDE requires second derivatives.

In [ ]:
class MLP(nn.Module):
    def __init__(self, input_dim=2, hidden_dim=64, hidden_layers=4):
        super().__init__()
        layers = [nn.Linear(input_dim, hidden_dim), nn.Tanh()]
        for _ in range(hidden_layers - 1):
            layers.extend([nn.Linear(hidden_dim, hidden_dim), nn.Tanh()])
        layers.append(nn.Linear(hidden_dim, 1))
        self.net = nn.Sequential(*layers)
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_normal_(module.weight)
                nn.init.zeros_(module.bias)

    def forward(self, inputs):
        lower = inputs.new_tensor([-1.0, 0.0])
        upper = inputs.new_tensor([1.0, 1.0])
        z = 2.0 * (inputs - lower) / (upper - lower) - 1.0
        return self.net(z)

model = MLP()


## 2. Collocation and constraint points

Interior points enforce the PDE. Initial points enforce the initial condition. Boundary points enforce both Dirichlet boundaries.

In [ ]:
def uniform_points(n, low, high):
    return low + (high - low) * torch.rand(n, 1)

n_f, n_0, n_b = 4000, 800, 800
X_f = torch.cat([uniform_points(n_f,-1,1), uniform_points(n_f,0,1)], 1)
x0 = uniform_points(n_0,-1,1)
X_0 = torch.cat([x0, torch.zeros_like(x0)], 1)
tb = uniform_points(n_b,0,1)
X_left = torch.cat([-torch.ones_like(tb),tb],1)
X_right = torch.cat([torch.ones_like(tb),tb],1)


## 3. Automatic differentiation and PDE residual

For $r_\theta=u_t-\alpha u_{xx}$, the first derivative graph must remain differentiable because $u_x$ is differentiated again to obtain $u_{xx}$.


In [ ]:
def heat_residual(model, xt, alpha):
    xt = xt.clone().detach().requires_grad_(True)
    u = model(xt)
    du = torch.autograd.grad(u, xt, torch.ones_like(u), create_graph=True, retain_graph=True)[0]
    u_x, u_t = du[:,0:1], du[:,1:2]
    d2 = torch.autograd.grad(u_x, xt, torch.ones_like(u_x), create_graph=True, retain_graph=True)[0]
    u_xx = d2[:,0:1]
    return u_t - alpha*u_xx

r = heat_residual(model, X_f, alpha)
print(r.shape, r.requires_grad)


## 4. Composite PINN loss

$$\mathcal L=\lambda_f\mathcal L_f+\lambda_{ic}\mathcal L_{ic}+\lambda_{bc}\mathcal L_{bc}.$$


In [ ]:
def loss_components(model):
    r = heat_residual(model, X_f, alpha)
    physics = r.square().mean()
    target0 = torch.sin(torch.pi*X_0[:,0:1])
    initial = (model(X_0)-target0).square().mean()
    boundary = 0.5*(model(X_left).square().mean()+model(X_right).square().mean())
    total = physics + 10.0*initial + 10.0*boundary
    return total, physics, initial, boundary


## 5. Adam training

In [ ]:
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
history = {k:[] for k in ['total','physics','initial','boundary']}
for epoch in range(1,2501):
    opt.zero_grad(set_to_none=True)
    total, physics, initial, boundary = loss_components(model)
    total.backward()
    opt.step()
    for k,v in zip(history,(total,physics,initial,boundary)):
        history[k].append(float(v.detach()))
    if epoch == 1 or epoch % 500 == 0:
        print(epoch, *(f'{history[k][-1]:.3e}' for k in history))


In [ ]:
plt.figure(figsize=(8,4))
for k,v in history.items():
    plt.semilogy(v,label=k)
plt.xlabel('epoch'); plt.ylabel('loss'); plt.legend(); plt.show()


## 6. Analytical validation

The exact solution is $e^{-\alpha\pi^2t}\sin(\pi x)$. It is used only for validation, not training.

In [ ]:
x = torch.linspace(-1,1,200); t = torch.linspace(0,1,120)
xx,tt = torch.meshgrid(x,t,indexing='ij')
grid = torch.stack([xx.reshape(-1),tt.reshape(-1)],1)
with torch.no_grad():
    pred = model(grid)
exact = torch.exp(-alpha*torch.pi**2*grid[:,1:2])*torch.sin(torch.pi*grid[:,0:1])
err = pred-exact
print('RMSE:', torch.sqrt(err.square().mean()).item())
print('max abs error:', err.abs().max().item())


In [ ]:
p = pred.reshape(len(x),len(t)).numpy()
e = exact.reshape(len(x),len(t)).numpy()
d = err.abs().reshape(len(x),len(t)).numpy()
fig,axes=plt.subplots(1,3,figsize=(15,4))
for ax,field,title in zip(axes,[p,e,d],['PINN','Exact','Absolute error']):
    im=ax.imshow(field,origin='lower',aspect='auto',extent=[0,1,-1,1])
    ax.set_xlabel('t'); ax.set_ylabel('x'); ax.set_title(title); fig.colorbar(im,ax=ax)
plt.tight_layout(); plt.show()


## 7. Gradient sanity check

A PDE derivative loss does not require every individual parameter to have a non-`None` gradient. The robust invariant is that the objective remains differentiable and backpropagates to parameters that influence it, with finite gradients.

In [ ]:
opt.zero_grad(set_to_none=True)
total, *_ = loss_components(model)
total.backward()
for name,p in model.named_parameters():
    print(name, p.grad is not None, None if p.grad is None else float(p.grad.norm()))
